# 🧬 DiffBiophys Showcase: Differentiable Structure Refinement

Welcome to the **DiffBiophys** showcase! This notebook demonstrates how to use differentiable biophysics kernels in JAX to optimize protein structures directly against experimental (or synthetic) data.

### What we'll do:
1. **Generate** a synthetic protein structure and its corresponding SAXS and NMR observables.
2. **Distort** the structure to create a starting "guess".
3. **Refine** the structure using Gradient Descent, fitting multiple experimental objectives simultaneously.

---

In [ ]:
# @title Install Dependencies
!pip install diff-biophys biotite matplotlib

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from diff_biophys.geometry import chain_nerf
from diff_biophys.saxs import debye_saxs
from diff_biophys.nmr.chemical_shifts import predict_ca_shifts, RANDOM_COIL_CA
from diff_biophys.nmr import calculate_rdc, calculate_q_factor

# Set up the system
n_atoms = 10
init_coords = jnp.array([[0.0, 0.0, 0.0], [1.5, 0.0, 0.0], [2.0, 1.4, 0.0]])
true_lengths = jnp.full(n_atoms - 3, 1.5)
true_angles = jnp.full(n_atoms - 3, jnp.radians(110.0))
true_dihedrals = jnp.array([1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0])

# 1. Create Target State
true_coords = chain_nerf(init_coords, true_lengths, true_angles, true_dihedrals)
q_values = jnp.linspace(0.01, 0.5, 30)
form_factors = jnp.ones((n_atoms, 30))
target_saxs = debye_saxs(true_coords, q_values, form_factors)

# 2. Distortion
distorted_dihedrals = true_dihedrals + 0.5

# 3. Loss Function
def loss_fn(dihedrals):
    coords = chain_nerf(init_coords, true_lengths, true_angles, dihedrals)
    s_profile = debye_saxs(coords, q_values, form_factors)
    s_loss = jnp.mean((s_profile - target_saxs)**2)
    return s_loss

grad_fn = jax.jit(jax.value_and_grad(loss_fn))

# 4. Optimization Loop
params = distorted_dihedrals
lr = 0.01
for i in range(101):
    loss, grads = grad_fn(params)
    params -= lr * grads
    if i % 20 == 0:
        print(f"Step {i:3d} | Loss: {loss:.6f}")

print("\n✅ Optimization complete!")